# 실습 내용

- 데이터 : titanic_train.csv
- kNN 알고리즘 사용

# 1.환경 준비

- 기본 **라이브러리**와 대상 **데이터**를 가져와 이후 과정을 준비합니다.

In [147]:
# 라이브러리 불러오기
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')
%config InlineBackend.figure_format = 'retina'

In [148]:
# 데이터 읽어오기
path = 'data/titanic_train.csv'
tt = pd.read_csv(path)

tt

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S
...,...,...,...,...,...,...,...,...,...,...,...,...
886,887,0,2,"Montvila, Rev. Juozas",male,27.0,0,0,211536,13.0000,NaN,S
887,888,1,1,"Graham, Miss. Margaret Edith",female,19.0,0,0,112053,30.0000,B42,S
888,889,0,3,"Johnston, Miss. Catherine Helen ""Carrie""",female,NaN,1,2,W./C. 6607,23.4500,NaN,S
889,890,1,1,"Behr, Mr. Karl Howell",male,26.0,0,0,111369,30.0000,C148,C


# 2.데이터 이해

- 분석할 데이터를 **충분히 이해**할 수 있도록 다양한 **탐색** 과정을 수행합니다.

In [149]:
# 상/하위 몇 개 행 확인
tt.head()

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S


In [150]:
# 하위 몇 개 행 확인
tt.tail()

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
886,887,0,2,"Montvila, Rev. Juozas",male,27.0,0,0,211536,13.00,NaN,S
887,888,1,1,"Graham, Miss. Margaret Edith",female,19.0,0,0,112053,30.00,B42,S
888,889,0,3,"Johnston, Miss. Catherine Helen ""Carrie""",female,NaN,1,2,W./C. 6607,23.45,NaN,S
889,890,1,1,"Behr, Mr. Karl Howell",male,26.0,0,0,111369,30.00,C148,C
890,891,0,3,"Dooley, Mr. Patrick",male,32.0,0,0,370376,7.75,NaN,Q


In [151]:
# 변수 확인
tt.info()

<class 'pandas.DataFrame'>
RangeIndex: 891 entries, 0 to 890
Data columns (total 12 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   PassengerId  891 non-null    int64  
 1   Survived     891 non-null    int64  
 2   Pclass       891 non-null    int64  
 3   Name         891 non-null    str    
 4   Sex          891 non-null    str    
 5   Age          714 non-null    float64
 6   SibSp        891 non-null    int64  
 7   Parch        891 non-null    int64  
 8   Ticket       891 non-null    str    
 9   Fare         891 non-null    float64
 10  Cabin        204 non-null    str    
 11  Embarked     889 non-null    str    
dtypes: float64(2), int64(5), str(5)
memory usage: 83.7 KB


In [152]:
# 기술통계 확인
tt.describe()

,PassengerId,Survived,Pclass,Age,SibSp,Parch,Fare
count,891.000000,891.000000,891.000000,714.000000,891.000000,891.000000,891.000000
mean,446.000000,0.383838,2.308642,29.699118,0.523008,0.381594,32.204208
std,257.353842,0.486592,0.836071,14.526497,1.102743,0.806057,49.693429
min,1.000000,0.000000,1.000000,0.420000,0.000000,0.000000,0.000000
25%,223.500000,0.000000,2.000000,20.125000,0.000000,0.000000,7.910400
50%,446.000000,0.000000,3.000000,28.000000,0.000000,0.000000,14.454200
75%,668.500000,1.000000,3.000000,38.000000,1.000000,0.000000,31.000000
max,891.000000,1.000000,3.000000,80.000000,8.000000,6.000000,512.329200


In [153]:
# 상관관계 확인
tt.corr(numeric_only=True)

,PassengerId,Survived,Pclass,Age,SibSp,Parch,Fare
PassengerId,1.000000,-0.005007,-0.035144,0.036847,-0.057527,-0.001652,0.012658
Survived,-0.005007,1.000000,-0.338481,-0.077221,-0.035322,0.081629,0.257307
Pclass,-0.035144,-0.338481,1.000000,-0.369226,0.083081,0.018443,-0.549500
Age,0.036847,-0.077221,-0.369226,1.000000,-0.308247,-0.189119,0.096067
SibSp,-0.057527,-0.035322,0.083081,-0.308247,1.000000,0.414838,0.159651
Parch,-0.001652,0.081629,0.018443,-0.189119,0.414838,1.000000,0.216225
Fare,0.012658,0.257307,-0.549500,0.096067,0.159651,0.216225,1.000000


# 3.데이터 전처리

- **전처리** 과정을 통해 머신러닝 알고리즘에 사용할 수 있는 형태의 데이터를 준비합니다.

**1) 변수 추가**
- 분석에 의미가 있다고 판단되는 변수를 추가합니다.

In [154]:
# title 변수 추가
tt['Title'] = tt['Name'].apply(
    lambda x: next((i for i in ['Mr.', 'Mrs.', 'Miss.', 'Master.'] if i in x), 'Others')
)

tt[['Title']]

,Title
0,Mr.
1,Mrs.
2,Miss.
3,Mrs.
4,Mr.
...,...
886,Others
887,Miss.
888,Miss.
889,Mr.


**2) 변수 제거**

- 분석에 의미가 없다고 판단되는 변수는 제거합니다.

In [155]:
# 제거 대상: PassengerId, Name, Ticket, Cabin
tt.drop(columns=['PassengerId', 'Name', 'Ticket', 'Cabin'], inplace=True)
# 확인
tt.head()

,Survived,Pclass,Sex,Age,SibSp,Parch,Fare,Embarked,Title
0,0,3,male,22.0,1,0,7.2500,S,Mr.
1,1,1,female,38.0,1,0,71.2833,C,Mrs.
2,1,3,female,26.0,0,0,7.9250,S,Miss.
3,1,1,female,35.0,1,0,53.1000,S,Mrs.
4,0,3,male,35.0,0,0,8.0500,S,Mr.


**3) 결측치 처리**

- 결측치가 있으면 제거하거나 적절한 값으로 채웁니다.

In [156]:
tt.isna().sum()

Survived      0
Pclass        0
Sex           0
Age         177
SibSp         0
Parch         0
Fare          0
Embarked      2
Title         0
dtype: int64

In [157]:
# Age 결측치를 Title 별 중앙값으로 채우기
tt['Age'] = tt.groupby('Title', group_keys=False)['Age'].apply(lambda x: x.fillna(x.median()))
tt.isna().sum()

Survived    0
Pclass      0
Sex         0
Age         0
SibSp       0
Parch       0
Fare        0
Embarked    2
Title       0
dtype: int64

In [158]:
# Embarked를 최빈값인 'S'로 채우기
tt['Embarked'] = tt['Embarked'].fillna('S')
print(tt['Embarked'].value_counts())
tt.isna().sum()

Embarked
S    646
C    168
Q     77
Name: count, dtype: int64


Survived    0
Pclass      0
Sex         0
Age         0
SibSp       0
Parch       0
Fare        0
Embarked    0
Title       0
dtype: int64

**3) x, y 분리**

- 우선 target 변수를 명확히 지정합니다.
- target을 제외한 나머지 변수들 데이터는 x로 선언합니다.
- target 변수 데이터는 y로 선언합니다.
- 이 결과로 만들어진 x는 데이터프레임, y는 시리즈가 됩니다.
- 이후 모든 작업은 x, y를 대상으로 진행합니다.

In [159]:
# target 확인
target = 'Survived'
x = tt.drop(target, axis=1)
y = tt[target]

# 데이터 분리


**4) 가변수화**

- 범주형 변수를 가변수화 합니다.

- drop_first=True 로 설정하면 새로 생성되는 열 중 하나(맨 앞)를 제거할 수 있습니다.
- 다중공선성 해결을 위한 옵션
- 성능이 좋아지고 안 좋아지고는 **시도** 해보지 않고는 알 수 없다.

In [172]:
# 가변수화 대상: Pclass, Sex, Embarked
tt = pd.get_dummies(x, columns=['Pclass', 'Sex', 'Embarked', 'Title'], drop_first=True, dtype=int)
tt.head()
# 가변수화
# 확인


,Age,SibSp,Parch,Fare,Pclass_2,Pclass_3,Sex_male,Embarked_Q,Embarked_S,Title_Miss.,Title_Mr.,Title_Mrs.,Title_Others
0,22.0,1,0,7.2500,0,1,1,0,1,0,1,0,0
1,38.0,1,0,71.2833,0,0,0,0,0,0,0,1,0
2,26.0,0,0,7.9250,0,1,0,0,1,1,0,0,0
3,35.0,1,0,53.1000,0,0,0,0,1,0,0,1,0
4,35.0,0,0,8.0500,0,1,1,0,1,0,1,0,0


**6) 학습용, 평가용 데이터 분리**

- 학습용, 평가용 데이터를 적절한 비율로 분리합니다.
- 반복 실행 시 동일한 결과를 얻기 위해 random_state 옵션을 지정합니다.

In [173]:
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=1)

# 4.모델링

- 본격적으로 모델을 **선언**하고 **학습**하고 **평가**하는 과정을 진행합니다.
- 우선 **회귀** 문제인지 **분류** 문제인지 명확히 구분합니다.

- 회귀 문제 인가요? 분류 문제인가요?
- 회귀인지 분류인지에 따라 사용할 알고리즘과 평가 방법이 달라집니다.
- 다음 알고리즘 사용
    - 알고리즘: KNeighborsClassifier

In [174]:
# 1단계: 불러오기
from sklearn.neighbors import KNeighborsClassifier

In [175]:
# 2단계: 선언하기
model = KNeighborsClassifier()

In [181]:
# 3단계: 학습하기
model.fit(X_train, y_train)

ValueError: could not convert string to float: 'Miss'

In [ ]:
# 4단계: 예측하기


# 5.분류 성능 평가

- 다양한 성능 지표로 분류 모델 성능을 평가합니다.

**1) Confusion Matrix**

In [ ]:
# 모듈 불러오기


# 성능 평가


In [ ]:
# 혼동행렬 시각화


**2) Accuracy**

In [ ]:
# 모듈 불러오기


# 성능 평가


**3) Precision**

In [ ]:
# 모듈 불러오기

# 성능 평가


**4) Recall**

In [ ]:
# 모듈 불러오기


# 성능 평가


**5) F1-Score**

In [ ]:
# 모듈 불러오기


# 성능 평가


**6) Classification Report**

In [ ]:
# 모듈 불러오기


# 성능 평가
